# SpecTexNet on Kaggle

**Before running** (right-hand panel → *Settings*):
- Accelerator: **GPU T4 x2** (do not use P100; recent PyTorch builds no longer support it)
- Internet: **On** (needs a phone-verified Kaggle account)
- *Add Input* → search **"fabric defects dataset" by nexuswho** → Add

**How sessions work.** Set `BATCH` below, then click **Save Version → Save & Run All (Commit)**.
The run continues in the background (max 12 h) even if you close the browser. When it finishes,
its `/kaggle/working` folder becomes the notebook's *Output*. For the next batch: *Add Input →
Your Work → Notebooks → this notebook* (the previous version's output), change `BATCH`, commit again.
Cell 3 copies the previous results back automatically, and finished folds are skipped.

In [ ]:
# ===== 1. SETTINGS: change only this cell =====
GITHUB_URL = "https://github.com/CodeWithYeasin/FabricDefectDetection.git"   # your repo
BATCH = 1          # 1, then 2, then 3 (one committed run each; rerun the same number if it ran out of time)
HOURS = 10.5       # no new fold starts after this many hours (Kaggle's limit is 12)

BATCHES = {
    1: ["spectex_convnext_tiny", "convnext_tiny", "spectex_resnet50", "resnet50",
        "spectex_efficientnet_b0", "efficientnet_b0", "mobilenetv3_large_100", "swin_tiny", "deit_small"],
    2: ["abl_no_spectral", "abl_concat_fusion", "abl_fixed_sr", "abl_single_scale", "abl_aug_flip",
        "abl_aug_naive_rot", "abl_loss_ce", "abl_no_derived", "abl_colour"],
    3: [],   # no CV training: efficiency, figures, statistics, LaTeX tables, deployment model
}

In [ ]:
# ===== 2. code + environment =====
import os, glob, subprocess, shutil, json, time
os.chdir("/kaggle/working")
if not os.path.exists("FabricDefectDetection"):
    !git clone -q $GITHUB_URL FabricDefectDetection
%cd /kaggle/working/FabricDefectDetection
!git pull -q
!pip install -q timm safetensors
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
# ===== 3. restore results of earlier batches (if a previous version's output was added as input) =====
prev = sorted(glob.glob("/kaggle/input/**/FabricDefectDetection/results", recursive=True))
for p in prev:
    print("restoring from", p)
    !rsync -a --ignore-existing "{p}/" results/
    d = os.path.dirname(p) + "/data"
    for f in ["splits.csv", "dedup_report.json", "duplicate_pairs.csv", "name_label_conflicts.csv",
              "image_stats_by_class.csv", "skipped.csv"]:
        if os.path.exists(f"{d}/{f}"):
            os.makedirs("data", exist_ok=True); shutil.copy(f"{d}/{f}", f"data/{f}")
    if os.path.exists(os.path.dirname(p) + "/paper"):
        !rsync -a "{os.path.dirname(p)}/paper/" paper/
print("experiments present:", sorted(os.listdir("results")) if os.path.exists("results") else [])

In [ ]:
# ===== 4. data: audit, de-duplicate, folds, cache (first batch only; later batches re-create the cache) =====
roots = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/hole/*.jpg", recursive=True)})
assert roots, "Dataset not found: Add Input -> 'fabric defects dataset' (nexuswho)"
DATA_ROOT = os.path.commonpath(roots); print("dataset root:", DATA_ROOT)

if not os.path.exists("data/splits.csv"):
    !python scripts/prepare_data.py --roots "$DATA_ROOT" --out data
    # leakage study: mimic the original (v1) protocol on purpose
    !python scripts/prepare_data.py --roots "$DATA_ROOT" --out data_leaky --derived keep --policy keep_all --no-groups --max-aspect 1000
else:
    # splits come from the earlier batch (identical folds); only rebuild the resized image cache
    import sys; sys.path.insert(0, "src")
    import pandas as pd
    from pathlib import Path
    from fabricnet.data.cache import cache_image
    df = pd.read_csv("data/splits.csv")
    for p, op in zip(df["path"], df["orig_path"]):
        if not Path(p).exists():
            cache_image(op, Path(p))
    print("cache ready:", len(df), "images")
print(json.dumps({k: v for k, v in json.load(open("data/dedup_report.json")).items()
                  if k in ["n_raw", "derived_copies", "nonphoto_dropped", "n_after", "name_label_conflicts",
                           "duplicate_groups", "clean_class_counts", "train_only_derived_counts"]}, indent=1))

In [ ]:
# ===== 5. batch 1 only: shortcut audit, then the v1 pipeline in the BACKGROUND on CPU =====
# (the scratch boosting takes 2-3 CPU hours; it runs while both GPUs train in step 6)
legacy = None
if BATCH == 1:
    !python scripts/shortcut_check.py
    os.makedirs("results/_logs", exist_ok=True)
    cmd = "python scripts/run_legacy_v1.py"
    if os.path.exists("data_leaky/splits.csv"):
        cmd += " && python scripts/run_legacy_v1.py --prefix leaky_ --set data.splits_csv=data_leaky/splits.csv data.grayscale=false"
    legacy = subprocess.Popen(cmd, shell=True, stdout=open("results/_logs/legacy_v1.log", "w"), stderr=subprocess.STDOUT)
    print("v1 baseline started in background, log: results/_logs/legacy_v1.log")

In [ ]:
# ===== 6. train this batch on both GPUs (resumable) =====
exps = BATCHES[BATCH]
if exps:
    !python scripts/run_queue.py --hours $HOURS --exps {" ".join(exps)}
    !tail -n 3 results/_logs/*.log
if legacy is not None:
    print("waiting for the v1 baseline to finish ..."); legacy.wait()
    !tail -n 5 results/_logs/legacy_v1.log

In [ ]:
# ===== 7. batch 3: efficiency, figures, statistics, LaTeX tables, deployment model =====
if BATCH == 3:
    !python scripts/benchmark_efficiency.py
    !python scripts/figure_architecture.py
    !python scripts/figure_method.py
    !python scripts/explain.py --exp spectex_convnext_tiny --fold 0 --per-class 2
    !python scripts/train.py --exp spectex_convnext_tiny --full
    !python scripts/export_model.py --run results/spectex_convnext_tiny/full
# the summary/tables are refreshed after every batch so you can follow progress
!python scripts/aggregate.py --proposed spectex_convnext_tiny

In [ ]:
# ===== 8. progress report =====
s = json.load(open("results/summary.json"))
rows = sorted(((k, v) for k, v in s.items() if not k.endswith("__notta")), key=lambda kv: -kv[1]["cv"]["macro_f1"]["mean"])
print(f"{'experiment':34s} {'runs':>4s} {'macro-F1':>15s} {'bal.acc':>8s} {'acc':>7s}")
for k, v in rows:
    c = v["cv"]
    print(f"{k:34s} {v['n_runs']:4d} {100*c['macro_f1']['mean']:7.2f} ± {100*c['macro_f1']['std']:5.2f} "
          f"{100*c['balanced_accuracy']['mean']:8.2f} {100*c['accuracy']['mean']:7.2f}")
if os.path.exists("results/shortcut_check.json"):
    print("shortcut check:", json.load(open("results/shortcut_check.json")))

In [ ]:
# ===== 9. keep the output small (Kaggle output limit 20 GB): drop the image cache, keep results =====
!rm -rf data/cache data_leaky/cache results/_cache
!du -sh /kaggle/working/FabricDefectDetection/results
# Download: right panel -> Output -> FabricDefectDetection/results, paper/, deploy/